# Mushroom Confusion Diagnosis — Run 5, Grad-CAM, Run 6 (resolution)


## Objective

Run 3/4 showed mushroom stuck at ~78-85% val_accuracy with a large train/val gap (~99.9% vs
~84%) — textbook overfitting signature. Run 5 tested that hypothesis directly: add regularization
(stronger augmentation, weight decay, label smoothing) and see if the gap closes. It didn't. This
notebook documents Run 5, the corrected comparison against Run 4, and a Grad-CAM investigation
into *why* — which points at a different diagnosis than overfitting.

Kept separate from `07_training_run_log.ipynb` since this is a focused investigation with
generated diagnostic images, not a routine run log entry.


## What changed for Run 5

Three cheap, standard regularizers, all newly config-driven (both `build_loss` and
`build_optimizer` already accepted `**kwargs`, so no new plumbing needed beyond reading them
from `training_config` in `scripts/train_baseline.py`):

- `augmentation_preset: light → medium` in `configs/mushroom.yaml` (rotation + stronger color
  jitter, on top of the existing flip)
- `weight_decay: 0.05` (up from AdamW's implicit 0.01 default)
- `label_smoothing: 0.1` (softens target confidence in the cross-entropy loss)

Also fixed an unrelated infrastructure problem discovered during Run 4: `scripts/train_truba_cpu.sbatch`
now requests `--exclusive` — Run 4 shared its node with another job (`CPUAlloc=61` when we only
asked for 56) and epoch times swung between 2900-4760s as a result. Run 5 confirms the fix: epoch
times are flat at ~950s throughout (see the chart below).


![Mushroom Run 5 metrics](assets/mushroom_run5_metrics.png)


## Run 4 vs Run 5 — corrected comparison

The quick read from the W&B charts alone made Run 5 look roughly equivalent to Run 4. Pulling the
exact numbers from both runs' best epoch (by `val_macro_f1`) tells a clearer story:

| | Run 4 (light aug, no weight decay) | Run 5 (medium aug + weight_decay + label_smoothing) |
|---|---|---|
| Best epoch | 18 | 24 |
| val_accuracy | **84.80%** | 83.27% |
| val_macro_f1 | **0.8126** | 0.7911 |
| train_accuracy (same epoch) | 99.93% | 99.95% |
| train/val accuracy gap | ~15.1 pts | ~16.7 pts |

**Regularization made things slightly worse, not better, on every metric that matters, and the
train/val gap didn't shrink either** — train accuracy is still ~99.9%+ regardless of augmentation
strength or weight decay. `val_loss`'s absolute value did go up in Run 5 (1.5 vs 0.88 in Run 4),
but that's an artifact of label smoothing changing the loss function's floor (it never lets loss
approach zero, even for perfect predictions) — not a real regression, and not comparable across
the two runs directly. Accuracy and macro-F1 are what's comparable, and both point the same way.

**This is the actual finding that matters**: if regularization aimed at "the model is
memorizing the training set" doesn't move the needle at all, the working hypothesis
("overfitting due to insufficient regularization") is probably wrong, or at least incomplete.


## The confused-pairs pattern

Run 5's top-10 confused pairs (from `outputs/reports/mushroom_resnet50_eval_report.json`):

| true → predicted | count |
|---|---|
| Fomitopsis pinicola → Fomitopsis mounceae | 64 |
| Fomes fomentarius → Fomitopsis betulina | 31 |
| Amanita muscaria → Amanita persicina | 30 |
| Evernia prunastri → Evernia mesomorpha | 25 |
| Fomitopsis pinicola → Fomes fomentarius | 24 |
| Fomes fomentarius → Ganoderma applanatum | 21 |
| Parmelia sulcata → Hypogymnia physodes | 21 |
| Pleurotus pulmonarius → Pleurotus ostreatus | 21 |
| Xanthoria parietina → Vulpicida pinastri | 21 |
| Leccinum scabrum → Leccinum aurantiacum | 18 |

**Every single one of these pairs is a same-genus confusion**: *Fomitopsis* with *Fomitopsis*,
*Amanita* with *Amanita*, *Evernia* with *Evernia*, *Pleurotus* with *Pleurotus*, *Leccinum* with
*Leccinum* — the two *Fomitopsis pinicola* rows and the *Fomes fomentarius* → *Ganoderma
applanatum* row are all bracket/polypore fungi that look alike even to the model apparently
across genus lines too. This is not noise — it's a consistent, taxonomically coherent error
pattern, which is a very different signal than "the model overfit the training set" would
produce (that would look like more random, spread-out confusion, not concentrated on
closely-related species pairs).

This reframes the question from *"how do we stop the model from memorizing?"* to *"can a
224×224 ResNet-50 tell these specific look-alike species apart at all, and if not, why?"*
— which is exactly what Grad-CAM can help answer: is the model looking at the right part of the
image and still failing (a genuine fine-grained discrimination limit), or is it looking
somewhere irrelevant (a fixable data/pipeline issue)?


## Grad-CAM: where is the model looking when it gets these wrong?

Implemented in `src/explainability/gradcam.py` — a standard Grad-CAM (Selvaraju et al., 2017):
hook the last conv block (`model.backbone.layer4`), backprop the predicted class's score, weight
the activation channels by their gradients, and overlay the resulting heatmap on the input image.

For each of the top 5 confused pairs, loaded the Run 5 best checkpoint, ran inference over the
full validation set, found actual misclassified examples (`true_label == A, predicted_label ==
B`), and ran Grad-CAM on 3 random examples per pair. Red/yellow = where the model's prediction is
most sensitive to; blue = ignored.


![Grad-CAM: Fomitopsis pinicola misclassified as Fomitopsis mounceae](assets/gradcam_Fomitopsis_pinicola_Fomitopsis_mounceae.png)


![Grad-CAM: Fomes fomentarius misclassified as Fomitopsis betulina](assets/gradcam_Fomes_fomentarius_Fomitopsis_betulina.png)


![Grad-CAM: Amanita muscaria misclassified as Amanita persicina](assets/gradcam_Amanita_muscaria_Amanita_persicina.png)


![Grad-CAM: Evernia prunastri misclassified as Evernia mesomorpha](assets/gradcam_Evernia_prunastri_Evernia_mesomorpha.png)


![Grad-CAM: Fomitopsis pinicola misclassified as Fomes fomentarius](assets/gradcam_Fomitopsis_pinicola_Fomes_fomentarius.png)


## Reading the Grad-CAM results

**In all 15 examples across all 5 pairs, the model's attention sits squarely on the fungus or
lichen itself** — never on the bark, snow, moss, hand, or background. There is no example of the
model "cheating" by keying off an irrelevant cue. That rules out the most easily-fixable
explanation (a background/shortcut-learning artifact) and supports the taxonomic-confusion
reading: **the model is looking at exactly the right structures and still can't separate these
species.**

A secondary pattern in the confidence scores is worth noting: the *Fomitopsis pinicola ↔
mounceae* and *Evernia prunastri ↔ mesomorpha* pairs get consistently high-confidence wrong
predictions (0.76-0.96) with attention covering the whole fruiting body/thallus — the model isn't
uncertain, it's confidently wrong, which is consistent with these species being genuinely
close to indistinguishable at this resolution/scale. The *Fomes fomentarius ↔ Fomitopsis
betulina* pair instead shows lower, more varied confidence (0.41-0.96) and attention sometimes
concentrated on a smaller sub-region rather than the whole structure — a messier, more
borderline case, possibly a harder pair even for the correct-answer cases, or images where the
diagnostic surface (pore structure, which isn't very visible in bracket-fungus photos taken from
this angle) simply isn't captured by the photo.

**What this does and doesn't tell us:**
- Does *not* support: more/heavier regularization (Run 5 already tested this, no effect), or
  suspecting a background/data-leakage artifact (Grad-CAM shows none).
- Does support treating this as a genuine fine-grained visual discrimination problem, where the
  next things worth testing (in order) are things that affect how much visual detail the model
  can actually see and use, not things that fight overfitting:
  1. **Higher input resolution** (384×384, already benchmarked as an option in Sprint 3) — 224px
     may be discarding the fine surface/texture detail that separates e.g. *Fomitopsis pinicola*
     from *mounceae*.
  2. **A different architecture** (EfficientNet-B3, per the Sprint 4 research note) — compare
     whether it makes the *same* genus-level mistakes. If yes, this is a dataset/resolution
     ceiling, not a ResNet-50-specific weakness. If no, architecture matters more than expected
     here.
  3. Only after 1-2: consider whether these specific confused species need targeted extra data,
     rather than assuming more data across the board would help evenly.


## Run 6 — resolution experiment (224px → 384px)

Testing priority 1 from the conclusion above: does more input resolution recover the fine
surface detail these look-alike species need? Isolated as the *only* change from Run 4
(`configs/mushroom.yaml`: `image_size: 224 → 384`, augmentation/weight_decay/label_smoothing
reverted to Run 4's settings — light augmentation, no extra weight decay, no label smoothing).
Same command: `sbatch scripts/train_truba_cpu.sbatch configs/mushroom.yaml 30 "" 300`.

- Duration: 89055.0s (1 day 44 min) — epoch time roughly tripled vs 224px (~2870-3330s vs
  ~950s), as expected from ~2.94x more pixels per image
- **Best epoch: 24** (val_macro_f1=0.8511, val_accuracy=88.10%)
- train_accuracy 99.99% at the best epoch — still fully memorizing the training set;
  384px didn't reduce overfitting, it just raised the val ceiling alongside it


![Mushroom Run 6 (384px) metrics](assets/mushroom_run6_384px_metrics.png)


### 224px vs 384px — real improvement, but partial

| | Run 5 (224px, regularized) | Run 6 (384px, Run-4 settings) |
|---|---|---|
| val_accuracy | 83.27% | **88.10%** |
| val_macro_f1 | 0.7911 | **0.8511** |
| train_accuracy (best epoch) | 99.95% | 99.99% |
| epoch duration | ~950s | ~2900-3300s (~3x) |

Clear, genuine gain: **+4.8 points accuracy, +0.06 macro-F1**. Resolution was worth trying.

### But the same confused pairs — just less often, not gone

| pair | Run 5 (224px) count | Run 6 (384px) count | change |
|---|---|---|---|
| Fomitopsis pinicola → Fomitopsis mounceae | 64 | 40 | -37.5% |
| Amanita muscaria → Amanita persicina | 30 | 14 | -53.3% |
| Fomes fomentarius → Fomitopsis betulina | 31 | 16 | -48.4% |
| Evernia prunastri → Evernia mesomorpha | 25 | 17 | -32.0% |
| Fomitopsis pinicola → Fomes fomentarius | 24 | 17 | -29.2% |
| Xanthoria parietina → Vulpicida pinastri | 21 | 18 | -14.3% |

Every one of Run 5's top pairs shows real improvement at 384px — resolution genuinely helps
separate these species, supporting the "fine detail lost at 224px" part of the diagnosis.

**But the pattern didn't disappear, it reshuffled.** Run 6's own top-10 confused pairs are
*still* exactly the same genus clusters (Fomitopsis, Evernia, Amanita, Xanthoria/Vulpicida),
plus a **new bidirectional pair** that wasn't prominent before: `Leccinum versipelle ↔
Leccinum aurantiacum` (17 and 16 misclassifications *in both directions*), and a third
Fomitopsis pair (`pinicola → betulina`, 15). `Fomitopsis mounceae` is still the single worst
class by F1 (0.504) even at 384px. Higher resolution didn't eliminate the genus-level ceiling,
it just moved where on that ceiling the remaining errors concentrate.

**Reading this**: resolution is a real, worthwhile lever, but not a full fix on its own — and
its cost is steep. At 300/class, full-dataset extrapolation (689,520 images, ~13.6x more) would
put a 384px run at roughly 2-3 *weeks* of wall-clock time on this CPU cluster, which isn't
practical, especially without mixed precision (still a no-op on CPU, see
`scripts/train_baseline.py` — this only matters on CUDA, not available here).


### Next: EfficientNet-B3 comparison, at 224px

Priority 3 from the original list, promoted up given 384px's cost/benefit: train
EfficientNet-B3 at 224px (not 384, to keep this experiment's cost controlled and isolate
architecture as the one variable) and check whether it makes the *same* genus-level mistakes.

- **If yes** (same Fomitopsis/Evernia/Amanita/Xanthoria confusions dominate): this is a
  dataset/subset-scale ceiling, not a ResNet-50-specific weakness — resolution and architecture
  both help partially but neither alone resolves it, and the real fix is more data specifically
  for these species, not more model changes.
- **If no** (EfficientNet separates these pairs meaningfully better): architecture matters more
  here than expected, worth pursuing further (e.g. EfficientNet at 384px too, if the accuracy
  gain justifies the cost).


## Full confusion matrix — and a correction to the genus-confusion narrative

Run 6's 384px checkpoint, full validation set (15,616 images), all 169 species — not just the
top-10 confused pairs this time. Computed with `scratch_confusion_matrix.py`, not saved to the
repo (one-off analysis, not a reusable script).

- **Species accuracy: 88.10%** — matches Run 6's reported number
- **Genus accuracy: 91.02%** (derived by mapping both true and predicted species to their genus)
- Of the **1,859 species-level errors**, only **456 (24.5%) still got the genus right**

**This is a real correction to the earlier framing.** The top-10 confused-pairs list (used in
every run's analysis so far) is dominated by same-genus pairs because those individual pairs
have the highest counts — but that list is not representative of errors *in aggregate*. Looking
at all 1,859 errors, **75.5% are actually cross-genus mistakes**, not within-genus ones. The
genus-clustering story was real but overstated: same-genus confusion is the single most visible
pattern, not the dominant one.

This matters directly for the hierarchical-loss hypothesis below: if only ~24.5% of species
errors are within-genus, then even a hypothetically *perfect* genus classifier could only ever
fix, at best, a quarter of the current errors — the other three-quarters are mistakes between
genera that genus-awareness doesn't help with by construction.


![Mushroom full species-level confusion matrix, sorted by genus](assets/mushroom_confusion_matrix_species_full.png)


![Mushroom genus-level confusion matrix](assets/mushroom_confusion_matrix_genus.png)


## EfficientNet-B3 comparison — same genus-level mistakes, slightly worse overall

Trained EfficientNet-B3 at 224px (not 384, to isolate architecture and control cost) on both
datasets, identical settings to Run 4 (mushroom) and the standing flower config otherwise:

| | ResNet-50 | EfficientNet-B3 |
|---|---|---|
| Mushroom val_accuracy | **84.80%** (Run 4) | 83.68% |
| Mushroom val_macro_f1 | **0.8126** | 0.7983 |
| Flower val_accuracy | **98.80%** (Run 3) | 98.78% |
| Flower val_macro_f1 | **0.9882** | 0.9867 |

ResNet-50 comes out slightly ahead on both datasets — not the "newer architecture wins" result
one might expect by default. More importantly, EfficientNet-B3's top confused pairs on mushroom
are the **same genus clusters**: `Fomitopsis pinicola → Fomitopsis mounceae` (37),
`Xanthoria parietina → Vulpicida pinastri` (37), `Amanita muscaria → Amanita persicina` (32),
`Fomes fomentarius → Ganoderma applanatum` (30), `Evernia prunastri → Evernia mesomorpha` (21).

**This directly answers the question the comparison was designed to ask**: since a different
architecture produces the same confusion pattern, it isn't a ResNet-50-specific weakness — it's
a property of the dataset/subset at this scale. Combined with the confusion-matrix finding above,
switching architectures was not worth pursuing further; effort went into the hierarchical model
instead (below), built on ResNet-50 since it's the stronger baseline.


## Hierarchical (genus+species) multi-task model

Approach: one shared ResNet-50 backbone, two linear heads (`genus_head`, `species_head`),
`total_loss = genus_loss + species_loss` (`src/models/hierarchical_resnet.py`,
`scripts/train_hierarchical.py` — a separate training script since the two-logit-tensor output
doesn't fit the shared single-task `src/training/engine.py` path used everywhere else). Genus
labels are derived from species names (first word) via a lookup tensor built once at startup —
no dataset changes needed. Kept at Run 4's settings otherwise (224px, light augmentation, same
`ReduceLROnPlateau`/early-stopping patience) specifically to isolate the hierarchical loss as the
one changed variable, per the plan: confusion matrix first (above), then hierarchical loss with
the LR schedule untouched, and only tune the schedule further if this step showed a clear win.


![Hierarchical genus+species training metrics](assets/mushroom_hierarchical_metrics.png)


### Results — a small species-accuracy gain, but the confused pairs didn't improve

| | Run 4 (single-head, 224px) | Hierarchical (genus+species, 224px) |
|---|---|---|
| val_species_accuracy | 84.80% | 85.3% (best epoch 29) |
| val_species_macro_f1 | 0.8126 | 0.8156 |
| val_genus_accuracy | — (not modeled) | 88.7% (best epoch) |
| Fomitopsis pinicola → mounceae | 64 | **73 (worse)** |
| Xanthoria parietina → Vulpicida pinastri | 21 | **31 (worse)** |
| Amanita muscaria → persicina | — | 32 |

Species accuracy and macro-F1 both improved, but only marginally (+0.5pt accuracy, +0.003
macro-F1) — and **the flagship same-genus confused pairs got worse in raw count**, not better.
The hypothesis going in was "teaching the model genus explicitly will push its features to
separate genus-mates better" — that didn't clearly happen here.

**Why, in light of the confusion-matrix finding above**: genus accuracy plateaus at 88.7%
here too (`train_genus_loss` collapses to ~0.001 by epoch 15 — the auxiliary task overfits the
300/class subset just as hard as species did). An auxiliary task that itself overfits doesn't
give the shared backbone a cleaner signal to learn from. And since only ~24.5% of species errors
are within-genus in the first place, even a *working* hierarchical signal has a low ceiling on
how much it could help — most of the error is genus confusion or unrelated mistakes that
genus-awareness doesn't target by construction.

**Worst-10 species (hierarchical model)** — same names as every prior run (Fomitopsis mounceae,
Boletus reticulatus, Leccinum aurantiacum, Amanita persicina, …), consistent with a stable,
subset-scale ceiling rather than noise from any one run.

**Conclusion**: the hierarchical loss is not the fix. Combined with Run 5 (regularization: no
help), Run 6 (resolution: real but partial help), and EfficientNet-B3 (architecture: no help),
four different interventions have now been tried against this ceiling. The one lever not yet
tested at scale is the one the confusion-matrix analysis points at most directly — more real
data (the full 4,080/class dataset, not this 300/class subset) — since every architecture/loss/
regularization change so far has run into the same wall while data quantity was held fixed.
